In [3]:
pip install pandas numpy scikit-learn imbalanced-learn xgboost matplotlib

   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/48.9 MB 81.9 kB/s eta 0:09:58
   ---------------------------------------- 0.0/48.9 MB 196.9 kB/s eta 0:04:09
   ---------------------------------------- 0.2/48.9 MB 981.9 kB/s eta 0:00:50
   ---------------------------------------- 0.5/48.9 MB 2.0 MB/s eta 0:00:25
    --------------------------------------- 0.8/48.9 MB 2.6 MB/s eta 0:00:19
    --------------------------------------- 1.2/48.9 MB 3.2 MB/s eta 0:00:15
   - -------------------------------------- 1.6/48.9 MB 4.0 MB/s eta 0:00:12
   - -------------------------------------- 2.1/48.9 MB 4.5 MB/s eta 0:00:11
   - -------------------------------------- 2.3/48.9 MB 4.8 MB/s eta 0:00:10
   -- ------------------------------------- 2.9/48.9 MB 5.3 MB/s eta 0:00:09
   -- ---------

In [5]:
#importing the necessary libraries
import os
import time
import warnings
 
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             accuracy_score, roc_auc_score,
                             average_precision_score, confusion_matrix,
                             precision_recall_curve, roc_curve)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

In [9]:
DATA_PATH = "creditcard.csv"
OUT_DIR = "results"
SEED = 42
TEST_SIZE = 0.2
SUBSAMPLE_LEGIT = None

In [12]:
os.makedirs(OUT_DIR, exist_ok=True)
np.random.seed(SEED)


#loading the data

In [15]:
df = pd.read_csv(DATA_PATH)
if SUBSAMPLE_LEGIT:
    fraud = df[df["Class"] == 1]
    legit = df[df["Class"] == 0].sample(SUBSAMPLE_LEGIT, random_state=SEED)
    df = pd.concat([fraud, legit]).sample(frac=1, random_state=SEED)
 
n_total = len(df)
n_fraud = int(df["Class"].sum())
info = (f"Rows: {n_total}\nColumns: {df.shape[1]}\n"
        f"Fraud: {n_fraud} ({100 * n_fraud / n_total:.4f}%)\n"
        f"Legitimate: {n_total - n_fraud}\n"
        f"Missing values: {int(df.isna().sum().sum())}\n"
        f"Duplicate rows: {int(df.duplicated().sum())}\n"
        f"Subsample setting: {SUBSAMPLE_LEGIT}\n")
print(info)

Rows: 284807
Columns: 31
Fraud: 492 (0.1727%)
Legitimate: 284315
Missing values: 0
Duplicate rows: 1081
Subsample setting: None



In [17]:
plt.figure(figsize=(4.5, 3.5))
counts = df["Class"].value_counts().sort_index()
plt.bar(["Legitimate", "Fraud"], counts.values, color=["#4c72b0", "#c44e52"])
plt.yscale("log")
plt.ylabel("Transactions (log scale)")
plt.title("Class distribution")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/class_distribution.png", dpi=200)
plt.close()

In [21]:
#2. Preprocess: scale Time and Amount (V1-V28 are already PCA outputs)

In [23]:
X = df.drop(columns="Class")
y = df["Class"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED)
 
scaler = StandardScaler()
for col in ["Time", "Amount"]:
    X_train[col] = scaler.fit_transform(X_train[[col]])
    X_test[col] = scaler.transform(X_test[[col]])
    # separate scaler per column
    scaler = StandardScaler()
 
info += (f"Train size: {len(X_train)} (fraud {int(y_train.sum())})\n"
         f"Test size: {len(X_test)} (fraud {int(y_test.sum())})\n")
with open(f"{OUT_DIR}/dataset_info.txt", "w") as f:
    f.write(info)
 
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
spw = neg / pos  # scale_pos_weight for XGBoost

In [25]:
def make_model(name, weighted):
    if name == "Logistic Regression":
        return LogisticRegression(max_iter=1000, random_state=SEED,
                                  class_weight="balanced" if weighted else None)
    if name == "Random Forest":
        return RandomForestClassifier(
            n_estimators=100, max_depth=12, n_jobs=-1, random_state=SEED,
            class_weight="balanced_subsample" if weighted else None)
    if name == "XGBoost":
        return XGBClassifier(
            n_estimators=200, max_depth=6, learning_rate=0.1,
            tree_method="hist", n_jobs=-1, random_state=SEED,
            eval_metric="logloss",
            scale_pos_weight=spw if weighted else 1)
 
 
models = ["Logistic Regression", "Random Forest", "XGBoost"]
strategies = ["Baseline", "Class weighting", "SMOTE"]
 

In [27]:
# SMOTE is applied to the TRAINING set only (never the test set)
t0 = time.time()
X_sm, y_sm = SMOTE(random_state=SEED).fit_resample(X_train, y_train)
print(f"SMOTE done in {time.time() - t0:.1f}s -> {len(X_sm)} training rows")
 
rows, store = [], {}
for strat in strategies:
    for name in models:
        if strat == "SMOTE":
            Xt, yt, weighted = X_sm, y_sm, False
        else:
            Xt, yt, weighted = X_train, y_train, (strat == "Class weighting")
        model = make_model(name, weighted)
        t0 = time.time()
        model.fit(Xt, yt)
        train_time = time.time() - t0
        proba = model.predict_proba(X_test)[:, 1]
        pred = (proba >= 0.5).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
        rows.append({
            "Model": name, "Strategy": strat,
            "Accuracy": accuracy_score(y_test, pred),
            "Precision": precision_score(y_test, pred, zero_division=0),
            "Recall": recall_score(y_test, pred),
            "F1": f1_score(y_test, pred),
            "ROC_AUC": roc_auc_score(y_test, proba),
            "PR_AUC": average_precision_score(y_test, proba),
            "TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "Train_time_s": train_time,
        })
        store[(name, strat)] = (model, proba, pred)
        print(f"{name:20s} | {strat:16s} | F1={rows[-1]['F1']:.3f} "
              f"PR-AUC={rows[-1]['PR_AUC']:.3f} ({train_time:.1f}s)")
 
res = pd.DataFrame(rows)
res.round(4).to_csv(f"{OUT_DIR}/results.csv", index=False)

C:\Users\AYUSHMAAN\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\AYUSHMAAN\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\AYUSHMAAN\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\AYUSHMAAN\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\AYUSHMAAN\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid

SMOTE done in 1.2s -> 454902 training rows
Logistic Regression  | Baseline         | F1=0.724 PR-AUC=0.743 (0.7s)
Random Forest        | Baseline         | F1=0.863 PR-AUC=0.864 (32.8s)
XGBoost              | Baseline         | F1=0.857 PR-AUC=0.844 (4.0s)
Logistic Regression  | Class weighting  | F1=0.114 PR-AUC=0.716 (1.6s)
Random Forest        | Class weighting  | F1=0.829 PR-AUC=0.842 (26.4s)
XGBoost              | Class weighting  | F1=0.860 PR-AUC=0.877 (3.9s)
Logistic Regression  | SMOTE            | F1=0.109 PR-AUC=0.725 (2.9s)
Random Forest        | SMOTE            | F1=0.654 PR-AUC=0.831 (63.6s)
XGBoost              | SMOTE            | F1=0.718 PR-AUC=0.856 (6.8s)


In [28]:
# 3. Plots
# 3a. PR curves for each strategy
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True)
for ax, strat in zip(axes, strategies):
    for name in models:
        _, proba, _ = store[(name, strat)]
        p, r, _ = precision_recall_curve(y_test, proba)
        ap = average_precision_score(y_test, proba)
        ax.plot(r, p, label=f"{name} (AP={ap:.2f})")
    ax.set_title(strat)
    ax.set_xlabel("Recall")
    ax.legend(fontsize=7, loc="lower left")
axes[0].set_ylabel("Precision")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/pr_curves.png", dpi=200)
plt.close()

In [29]:
# 3b. ROC curves for each strategy
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True)
for ax, strat in zip(axes, strategies):
    for name in models:
        _, proba, _ = store[(name, strat)]
        fpr, tpr, _ = roc_curve(y_test, proba)
        auc = roc_auc_score(y_test, proba)
        ax.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=0.7)
    ax.set_title(strat)
    ax.set_xlabel("False positive rate")
    ax.legend(fontsize=7, loc="lower right")
axes[0].set_ylabel("True positive rate")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/roc_curves.png", dpi=200)
plt.close()
 

In [33]:
# 3c. Accuracy vs F1 vs PR-AUC (shows why accuracy misleads)
labels = [f"{r.Model.split()[0][:2]}-{r.Strategy.split()[0][:4]}"
          for r in res.itertuples()]
x = np.arange(len(res))
plt.figure(figsize=(10, 3.8))
plt.bar(x - 0.25, res["Accuracy"], 0.25, label="Accuracy")
plt.bar(x, res["F1"], 0.25, label="F1")
plt.bar(x + 0.25, res["PR_AUC"], 0.25, label="PR-AUC")
plt.xticks(x, labels, rotation=45, ha="right", fontsize=8)
plt.legend()
plt.title("Accuracy vs. F1 vs. PR-AUC")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/metric_comparison.png", dpi=200)
plt.close()
 

In [35]:
# 3d. Confusion matrix of best model by F1
best = res.sort_values("F1", ascending=False).iloc[0]
_, _, best_pred = store[(best["Model"], best["Strategy"])]
cm = confusion_matrix(y_test, best_pred)
plt.figure(figsize=(3.8, 3.4))
plt.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.xticks([0, 1], ["Legit", "Fraud"])
plt.yticks([0, 1], ["Legit", "Fraud"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"{best['Model']} + {best['Strategy']}", fontsize=9)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/confusion_matrix_best.png", dpi=200)
plt.close()

In [39]:
 #3e. Top-10 feature importances (XGBoost, class weighting)
xgb_model = store[("XGBoost", "Class weighting")][0]
imp = pd.Series(xgb_model.feature_importances_, index=X.columns)
imp = imp.sort_values().tail(10)
plt.figure(figsize=(5, 3.6))
imp.plot(kind="barh", color="#4c72b0")
plt.title("XGBoost top-10 feature importances")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/feature_importance.png", dpi=200)
plt.close()
 
print("\nBest by F1:", best["Model"], "+", best["Strategy"])
print(f"All outputs saved in ./{OUT_DIR}/")


Best by F1: Random Forest + Baseline
All outputs saved in ./results/
